In [1]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from keras.optimizers import Adam, RMSprop
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV
from joblib import dump, load

In [2]:
from numpy import cos, sin

In [3]:
def def_parametros():
    parametros = [
        "TIME",
        "TEMPO_SAD",
        "EVENTO",
        "HH",
        "MM",
        "SS",
        "MSS",
        "TOT",
        "ALFAG",
        "BETAG",
        "DL",
        "DM",
        "DN",
        "DF",
        "NX",
        "NY",
        "NZ",
        "P",
        "Q",
        "R",
        "PB",
        "PB1",
        "QB",
        "QB1",
        "TI",
        "T5_C",
        "TETAL",
        "FIL",
        "CID",
        "NG",
        "PSI",
        "PBEX",
        "PIX",
        "PIY",
        "VB",
        "VB1",
        "ZPB",
        "ZPB1",
        "W",
        "pa",
        "qc",
        "Zp",
        "Vc",
        "Mach",
        "Ta_C",
        "Va",
        "Alfa",
        "Beta",
        "Ta_K",
        "T",
        "No_OBS"
    ]
    return parametros

In [ ]:
files = ['039_2013_VIN_FND', '040_2013_ASO_VTO', '041_2013_GEO_AIR', '042_2013_VAU_FTR']
def make_df():
    local = 'C:\\Users\\ymarc\\Documents\\PMG\\Dados_AT26_VoosCompletos\\'

    parametros = def_parametros()

    all_dfs = []
    for file in files:
        data = loadmat(local + file + '.mat')
        df = pd.DataFrame(data['DADOS'])
        df.columns = parametros
        df['filename'] = file
        all_dfs.append(df)
    df = pd.concat(all_dfs, ignore_index=True)

    df['Z'] = 0.3048*df['Zp'] # Altitude em metros
    df['a1'] = df['Z']<=11000
    df['a2'] = df['Z']>11000

    df['Pa'] = (1013.25*(1-6.87559e-6*df['Zp'])**(5.25611))*(df['a1'])+(226.306*np.exp(157.696e-6 *(11000-df['Z'])))*(df['a2']) # Pressao em mbar
    df['Rho'] = df['Pa']/(2.869*(df['Ta_K'])) # Densidade em kg/m³

    S = 19.3
    # df['CL'] = 0.0756 * (df['AOA'] + 4.27) 
    df['CL'] = (2 * df['NZ'] * df['W'])/(df['Rho'] * df['Va']**2 * S)
    df['CD'] = 0.024 + 0.062 * df['CL']**2 # Conferir CL e CD depois!

    Ta0 = 288.15
    df['theta'] = df['Ta_K']/Ta0
    df['N1_over_sqrt_theta'] = df['NG']/np.sqrt(df['theta'])

    Pa0 = 1013.25
    df['delta'] = df['Pa']/Pa0
    df['T_over_delta'] = (df['T']*4.44822)/df['delta']

    df['D'] = df['Rho']*(df['Va']*0.5144)**2*S*df['CD']/2 # Arrasto em Newtons
    df['D_over_delta'] = df['D']/df['delta']

    df['faixa_altitude'] = None
    # A = df['Zp']/1000<50
    # B = (df['Zp']/1000>=50) & (df['Zp']/1000<150)
    # C = (df['Zp']/1000>=150) & (df['Zp']/1000<300)
    # D = df['Zp']/1000>=300
    df.loc[df['Zp']/1000<10, 'faixa_altitude'] = 'A'
    df.loc[(df['Zp']/1000>=10) & (df['Zp']/1000<20), 'faixa_altitude'] = 'B'
    df.loc[(df['Zp']/1000>=20) & (df['Zp']/1000<30), 'faixa_altitude'] = 'C'
    df.loc[df['Zp']/1000>=30, 'faixa_altitude'] = 'D'

    return df

In [140]:
def split_train_test_val(df):
    df['type'] = 'train'    
    df.loc[df['faixa_altitude'] == 'B', 'type'] = 'test'

    test_size = 0.2
    m = round((1-test_size)*len(df.loc[df['type'] == 'test']))
    val_indices = df.loc[df['type'] == 'test'].sample(n=m, random_state=42).index
    df.loc[val_indices, 'type'] = 'val'
    
    return df

In [141]:
def scale_data(df, features, targets):
    for param in features + targets:
        df[param + '_scaled'] = None

    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    for param in features:
        df.loc[df['type'] == 'train', param + '_scaled'] = scaler_x.fit_transform(df.loc[df['type'] == 'train'][[param]]).ravel()
        df.loc[df['type'] == 'test', param + '_scaled'] = scaler_x.transform(df.loc[df['type'] == 'test'][[param]]).ravel()
        df.loc[df['type'] == 'val', param + '_scaled'] = scaler_x.transform(df.loc[df['type'] == 'val'][[param]]).ravel()

    for param in targets:
        df.loc[df['type'] == 'train', param + '_scaled'] = scaler_y.fit_transform(df.loc[df['type'] == 'train'][[param]]).ravel()
        df.loc[df['type'] == 'test', param + '_scaled'] = scaler_y.transform(df.loc[df['type'] == 'test'][[param]]).ravel()
        df.loc[df['type'] == 'val', param + '_scaled'] = scaler_y.transform(df.loc[df['type'] == 'val'][[param]]).ravel()

    return df, scaler_x, scaler_y

In [142]:
def build_model(features, targets, architecture, hyperparameters):
    input_shape = (len(features), )

    model = keras.Sequential()

    targets_names = targets[0]
    for i in range(1, len(targets)):
        targets_names += '_' + targets[i]

    # Camadas ocultas
    for i in range(architecture['camadas_ocultas']):
        model.add(layers.Dense(
            architecture['neurons'], 
            input_shape=input_shape if i == 0 else None,
            activation=architecture['activation'],
            name=f'oculta_{i+1}_{targets_names}'
        ))
        
        if architecture['dropout'] > 0:
            model.add(layers.Dropout(architecture['dropout']))
    
    # Camada de saída
    model.add(layers.Dense(len(targets), name=f'saida_{targets_names}'))
    
    # Compilar modelo
    if hyperparameters['optimizer'] == 'adam':
        opt = Adam(learning_rate=hyperparameters['learning_rate'])
    else:
        opt = RMSprop(learning_rate=hyperparameters['learning_rate'])
    
    model.compile(loss="mae", optimizer=opt, metrics=["mae"])
    
    return model

In [143]:
def train_model(df, features, targets, model, hyperparameters):

    features_scaled = []
    for item in features:
        features_scaled.append(item + '_scaled')
    
    targets_scaled = []
    for item in targets:
        targets_scaled.append(item + '_scaled')

    X_train, X_val = np.array(df.loc[df['type'] == 'train', features_scaled]).astype(np.float32), np.array(df.loc[df['type'] == 'val', features_scaled]).astype(np.float32)
    Y_train, Y_val = np.array(df.loc[df['type'] == 'train', targets_scaled]).astype(np.float32), np.array(df.loc[df['type'] == 'val', targets_scaled]).astype(np.float32)

    callback = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=hyperparameters['patience']
    )

    training_history = model.fit(
        X_train, Y_train,
        epochs=hyperparameters['epochs'],
        batch_size=hyperparameters['batch_size'],
        callbacks=[callback],
        validation_data=(X_val, Y_val),
        verbose=0
        
    )

    return model, training_history

In [144]:
def pipeline(architecture, hyperparameters, features, targets):
    df = make_df()
    df = split_train_test_val(df)
    df, scaler_x, scaler_y = scale_data(df, features, targets)

    model = build_model(features, targets, architecture, hyperparameters)

    model, training_history = train_model(df, features, targets, model, hyperparameters)
    # training_history = 'mock'
    return df, scaler_x, scaler_y, model, training_history

In [145]:
def evaluate(model, df, features, targets, scaler_y):

    features_scaled = []
    for item in features:
        features_scaled.append(item + '_scaled')
    
    X_test = np.array(df.loc[df['type'] == 'test', features_scaled]).astype(np.float32)
    
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)

    for item in targets:
        df[item + '_pred'] = None

    for i, item in enumerate(targets):
        df.loc[df['type'] == 'test', item + '_pred'] = pred[:, i]

    real = np.array(df.loc[df['type'] == 'test', targets]).astype(np.float32)

    erro = (pred - real)* 100 / real
    avg_error = np.array([np.mean(erro[:,i]) for i in range(np.shape(erro)[1])]).astype(np.float32)

    results = {
        'predicted': pred,
        'real': real,
        'rel_error': erro,
        'avg_error': avg_error
    }
    
    return results

In [146]:
def analyze (features,
             targets,
             architecture={
                 'neurons': 60,
                 'camadas_ocultas': 2,
                 'activation': 'relu',
                 'dropout': 0.0
             },
             hyperparameters={
                 'optimizer': 'adam',
                 'learning_rate': 0.0001,
                 'epochs': 60,
                 'batch_size': 32,
                 'patience': 3
             }
            ):
    df, scaler_x, scaler_y, model, training_history = pipeline(
        architecture=architecture,
        hyperparameters=hyperparameters,
        features=features,
        targets=targets
    )

    targets_names = targets[0]
    for i in range(1, len(targets)):
        targets_names += '_' + targets[i]

    results = evaluate(model, df, features, targets, scaler_y)

    plt.plot(training_history.history['loss'])
    plt.plot(training_history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.grid()
    plt.savefig(f'{targets_names}_approach_training.png')
    plt.show()

    _, axs = plt.subplots(
        2 * len(targets),
        1,
        sharex=True,
        figsize=(12, 3 * len(targets)),
        constrained_layout=True
    )
    axs = axs.flatten()
    dot_size = 2

    for i, item in enumerate(targets):
        axs[2*i].set_title(f'Comparação de {item} Real vs. Predito')
        axs[2*i].scatter(df.loc[df['type'] == 'test']['No. OBS.'], df.loc[df['type']=='test'][f'{item}'], dot_size, label=f'{item} Predito')
        axs[2*i].scatter(df.loc[df['type'] == 'test']['No. OBS.'], df.loc[df['type']=='test'][f'{item}_pred'], dot_size, label=f'{item} Real')
        axs[2*i].legend()
        axs[2*i].grid()

        axs[2*i+1].set_title(f'Erro médio: {results["avg_error"][i]:.2f}%')
        axs[2*i+1].scatter(df.loc[df['type'] == 'test']['No. OBS.'], results["rel_error"][:,i], dot_size)
        axs[2*i+1].legend()
        axs[2*i+1].grid()
    plt.savefig(f'{targets_names}_approach_results.png')
    plt.show()

    return df, scaler_x, scaler_y, model, training_history, results